# Smart MCQ Solver

**Objective**: The goal of this competition is to build a machine learning pipeline capable of answering complex multiple-choice questions. 

Given a question prompt and five possible options (A, B, C, D, E), the model must rank the top 3 most likely answers. The evaluation metric for the competition is **Mean Average Precision at 3 (MAP@3)**, meaning models are rewarded heavily for getting the correct answer as their #1 prediction.

---

## Section 1: Project Setup & Data Ingestion
*Initializing the environment and loading the competition datasets.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load datasets (Using Kaggle Environment Paths)
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'


train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Training Set Shape: {train_df.shape}")
print(f"Test Set Shape: {test_df.shape}")

## Section 2: Exploratory Data Analysis (EDA)
*Checking for class imbalances and understanding text length distributions to inform our modeling strategy.*

In [ ]:
display(train_df.head(3))

In [ ]:
# Check answer distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=train_df, x='answer', order=['A', 'B', 'C', 'D', 'E'])
plt.title('Distribution of Correct Answers in Training Set')
plt.show()

In [ ]:
# Basic stats on prompt lengths
train_df['prompt_length'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
print("Prompt Word Count Stats:")
print(train_df['prompt_length'].describe())

## Section 3: Shared Utilities
*Foundational helper functions for evaluation and validation splits.*

### Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Create reproducible validation split
train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['answer'])
print(f"Train Split: {len(train_split)} rows")
print(f"Validation Split: {len(val_split)} rows")

### Evaluation Metric (MAP@3)

In [ ]:
def calculate_map3(predictions, ground_truth):
    """Calculates Mean Average Precision @ 3."""
    scores = []
    for pred_str, gt in zip(predictions, ground_truth):
        pred_list = pred_str.strip().split()
        score = 0.0
        num_correct = 0
        for k, label in enumerate(pred_list[:3], start=1):
            if label == gt:
                num_correct += 1
                score += num_correct / k
        scores.append(score)
    return np.mean(scores)

# Quick sanity check to ensure the metric works as expected
dummy_preds = ['A B C', 'D A E', 'C B A']
dummy_truths = ['A', 'A', 'C']
print(f"Sanity Check MAP@3 Score: {calculate_map3(dummy_preds, dummy_truths)}")